# 05 — Chemical similarity, and which direction it points

SEA nominates a drug–target pair from ligand similarity: the drug resembles known ligands of
the target. A natural check is whether that similarity predicts which nominations survive
the bench. It does not — and an earlier write-up of this work reported it pointing the wrong
way, which is worth pinning down because a sign error and a real inversion look identical in
a table.

In [1]:
import sys, os
from pathlib import Path

# Work from the repository root wherever the notebook is launched from.
_root = Path.cwd().resolve()
while not (_root / "src" / "offtarget").exists():
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, str(_root / "src"))

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score
from offtarget.pipeline import sea_baseline
from offtarget.metrics import stratified_auc

pd.set_option("display.width", 140)
sea = sea_baseline(Path("data/raw/Predictions.dat"), Path("data/raw/pid_confirmation_status.dat"))
assayed = sea[sea.label.notna()].dropna(subset=["Max Tc", "neg_log_evalue"])
print(f"assayed predictions: {len(assayed)}   "
      f"confirmed: {int((assayed.label == 'assay_active').sum())}")

assayed predictions: 1023   confirmed: 196


## SEA's own scores against its own outcomes

One source, one Tc column, one E-value column, so there is nothing to mismatch.

In [2]:
rows = []
for c in ["Max Tc", "neg_log_evalue", "Charge Probability"]:
    d = assayed.dropna(subset=[c])
    yy = (d.label == "assay_active").astype(int)
    auc, used = stratified_auc(d, c)
    rows.append({"score": c, "n": len(d),
                 "pooled AUC": round(roc_auc_score(yy, d[c]), 3),
                 "within-target AUC": round(auc, 3), "targets used": used})
pd.DataFrame(rows)

,score,n,pooled AUC,within-target AUC,targets used
0,Max Tc,1023,0.536,0.536,39
1,neg_log_evalue,1023,0.510,0.580,39
2,Charge Probability,997,0.689,0.617,39


Max Tc reaches 0.536 within target, the E-value 0.580. Both are close to chance and both are
**above** it. Chemical similarity to known ligands is barely informative about whether the
nominated interaction is real — but it is not anti-informative.

## Where 0.104 came from

An earlier write-up reported a SEA Tc AUC of 0.104. An AUC of 0.104 is a strongly
*anti*-predictive score: it would mean the more a drug resembles known ligands of a target,
the less likely it is to bind it. That is not a finding, it is a bug — and the source of it
is visible in the file that number came from.

In [3]:
xl = pd.ExcelFile("data/boltz_outputs/boltzresults_individual.xlsx")
true_sea, false_sea = xl.parse("TrueSea"), xl.parse("FalseSea")
print("TrueSea  similarity columns:",
      [c for c in true_sea.columns if "Tc" in str(c)])
print("FalseSea similarity columns:",
      [c for c in false_sea.columns if "Tc" in str(c)])

TrueSea  similarity columns: ['ECFP_4 Tc', 'Combined Tc']
FalseSea similarity columns: ['Max Tc']


The actives sheet carries `ECFP_4 Tc` and `Combined Tc`. The inactives sheet carries
`Max Tc`. There is no column common to both, so any AUC computed across them compared two
different quantities — and `Combined Tc` and `Max Tc` are on different scales.

In [4]:
pd.DataFrame({
    "Combined Tc (actives sheet)": true_sea["Combined Tc"].describe(),
    "Max Tc (inactives sheet)": false_sea["Max Tc"].describe(),
}).round(3)

,Combined Tc (actives sheet),Max Tc (inactives sheet)
count,130.000,534.000
mean,0.610,0.611
std,0.148,0.135
min,0.143,0.133
25%,0.520,0.549
50%,0.630,0.618
75%,0.700,0.688
max,0.960,1.000


The medians differ by enough to produce an AUC near zero on their own, whichever way the
classes fall. **The inversion was an artefact of joining two different columns**, and the
repository does not carry the 0.104 figure anywhere.

## The honest version

Within target, SEA's similarity score has an AUC of 0.536 for distinguishing its own
confirmed predictions from its own disproved ones. That is a much less exciting sentence
than an inversion, and it is the one the data supports.

In [5]:
per = assayed.groupby("Target").agg(n=("Max Tc", "size"),
                                    active=("label", lambda s: (s == "assay_active").sum()))
per = per[(per.active > 0) & (per.active < per.n)]
print(f"targets contributing both classes: {len(per)}")
per.sort_values("n", ascending=False).head(10)

targets contributing both classes: 39


,n,active
Target,,
ADRB2,51,9
OPRM,45,6
5HT2C,38,4
DRD1,36,7
OPRD,35,1
5HT2A,33,11
DRD2,33,3
5HT2B,31,8
ADA2C,29,9
